In [1]:
import pandas as pd
import numpy as np

In [2]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

In [35]:
# read the data:
df = pd.read_csv("../data/external/India-Food-Delivery-Time-Prediction.csv")
df.head()

,Unnamed: 0,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,Time_Order_picked,Weatherconditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken(min)
0,0,0x4607,INDORES13DEL02,37,4.9,22.745049,75.892471,22.765049,75.912471,19-03-2022,11:33:33,11:45:29,conditions Sunny,High,2,Snack,motorcycle,0,No,Urban,(min) 24
1,1,0xb379,BANGRES18DEL02,34,4.5,12.913041,77.683237,13.043041,77.813237,25-03-2022,19:45:37,19:51:49,conditions Stormy,Jam,2,Snack,scooter,1,No,Metropolitian,(min) 33
2,2,0x5d6d,BANGRES19DEL01,23,4.4,12.914264,77.678400,12.924264,77.688400,19-03-2022,8:32:58,8:48:47,conditions Sandstorms,Low,0,Drinks,motorcycle,1,No,Urban,(min) 26
3,3,0x7a6a,COIMBRES13DEL02,38,4.7,11.003669,76.976494,11.053669,77.026494,05-04-2022,18:03:58,18:12:52,conditions Sunny,Medium,0,Buffet,motorcycle,1,No,Metropolitian,(min) 21
4,4,0x70a2,CHENRES12DEL01,32,4.6,12.972793,80.249982,13.012793,80.289982,26-03-2022,13:34:16,13:45:36,conditions Cloudy,High,1,Snack,scooter,1,No,Metropolitian,(min) 30


In [130]:
# Shape of the data:
df.shape

(41953, 21)

### Data Level data cleaning

observations:

- There are zero duplicates.
- All the rows are from feb, march, april 2022.
 
Dirty Data:    
1. Time_Ordered got 1600 null values.
2. We can drop ID column.
3. There NaN values in density age, rating, time_ordered, multiple_deliveries, city_type.
    
    
Messy Data:
1. change column names because they are too big
2. person_id got city names in it like BANG -> Banglore, etc.
3. weather column got "condition" in it with every row and target column got "(min)" in it.
4. The data_types are messed.
5. values have trailing spaces in the end.

In [4]:
df.sample(1)

,Unnamed: 0,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,Time_Order_picked,Weatherconditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken(min)
40140,40140,0xcd16,LUDHRES010DEL02,21,4.6,30.89286,75.822199,31.02286,75.952199,12-02-2022,19:34:48,19:40:22,conditions Fog,Jam,1,Meal,scooter,NaN,No,Urban,(min) 30


In [9]:
df.duplicated().sum()

0

In [76]:
df["Weatherconditions"] = df["Weatherconditions"].str.replace("conditions ", "").str.strip()

In [83]:
df["time"] = df["time"].str.split(")").str.get(1).str.strip()

In [82]:
df["id"] = df["id"].str.split("RES").str.get(0)
df.rename(columns={"id": "city"}, inplace=True)

### Column Type:

categorical cols: city,weather, 
traffi, 
conditi, 0
order_t,  0
vehicle_,   0
multi_deliv,  905
fe,   215
city_type        
Numerical: rest_lat, 
rest_lon, 
delivery_l, 0
del, timeivery_lo
datetime: date, ordered_time, picked_timeng 

In [95]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 40199 entries, 0 to 41952
Data columns (total 19 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   city              40199 non-null  object 
 1   age               40199 non-null  float32
 2   ratings           40155 non-null  object 
 3   rest_lat          40199 non-null  float64
 4   rest_long         40199 non-null  float64
 5   delivery_lat      40199 non-null  float64
 6   delivery_long     40199 non-null  float64
 7   date              40199 non-null  object 
 8   ordered_time      40153 non-null  object 
 9   picked_time       40199 non-null  object 
 10  weather           40153 non-null  object 
 11  traffic           40153 non-null  object 
 12  condition         40199 non-null  int64  
 13  order_type        40199 non-null  object 
 14  vehicle_type      40199 non-null  object 
 15  multi_deliveries  39352 non-null  object 
 16  festival          39994 non-null  object 
 17

In [96]:
cat_col = ["city", "weather", "traffic", "order_type", "vehicle_type", "multi_deliveries", "festival", "city_type"]

for col in cat_col:
    df[col] = df[col].str.strip()

C:\Users\RR\AppData\Local\Temp\ipykernel_4056\2060313890.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = df[col].str.strip()


In [78]:
# fix NaN values:
df.replace("NaN ", np.nan, inplace=True)

In [84]:
df.isnull().sum()

city                   0
age                 1719
ratings             1763
rest_lat               0
rest_long              0
delivery_lat           0
delivery_long          0
date                   0
ordered_time        1600
picked_time            0
weather              569
traffic              555
condition              0
order_type             0
vehicle_type           0
multi_deliveries     905
festival             215
city_type           1114
time                   0
dtype: int64

In [80]:
# Drop Columns
df.drop(columns=['Unnamed: 0', 'ID'], inplace=True)

In [81]:
# rename the column names:
df.rename(columns={"Delivery_person_ID": "id",
"Delivery_person_Age":"age",
"Delivery_person_Ratings":"ratings",
"Restaurant_latitude": "rest_lat",
"Restaurant_longitude": "rest_long",
"Delivery_location_latitude": "delivery_lat",
"Delivery_location_longitude": "delivery_long",
"Order_Date": "date",
"Time_Orderd": "ordered_time",
"Time_Order_picked": "picked_time",
"Weatherconditions": "weather",
"Road_traffic_density": "traffic",
"Vehicle_condition": "condition",
"Type_of_order": "order_type",
"Type_of_vehicle": "vehicle_type",
"multiple_deliveries": "multi_deliveries",
"Festival": "festival",
"City": "city_type",
"Time_taken(min)": "time"}, inplace=True)

### column-wise observations:

observations:

1. 

In [85]:
cities = df["city"].value_counts()
len(cities)

22

In [86]:
df["city"].isnull().sum()

0

In [87]:
df["age"].isnull().sum()

1719

In [161]:
df["age"] = df["age"].astype(np.float32)

In [89]:
df["age"].describe()

count    40234.000000
mean        29.563330
std          5.812417
min         15.000000
25%         25.000000
50%         30.000000
75%         35.000000
max         50.000000
Name: age, dtype: float64

In [162]:
df[df["age"] < 18].shape

(35, 19)

In [165]:
df.shape

(41918, 19)

In [163]:
df = df.drop(index = df[df["age"] < 18].index)
df.shape

(41918, 19)

In [164]:
df["ratings"] = df["ratings"].astype(np.float32)

In [68]:
df["ratings"].describe()

count    40155.000000
mean         4.635552
std          0.318063
min          2.500000
25%          4.500000
50%          4.700000
75%          4.900000
max          6.000000
Name: ratings, dtype: float64

In [166]:
df[df["ratings"] > 5].shape

(46, 19)

In [152]:
df.shape

(41918, 19)

In [171]:
df = df.drop(index = df[df["ratings"] > 5].index)
df.shape

(41872, 19)

In [73]:
df["ratings"].isnull().sum()

0

In [26]:
df["date"] = pd.to_datetime(df["date"])

C:\Users\RR\AppData\Local\Temp\ipykernel_11040\3228555721.py:1: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["date"] = pd.to_datetime(df["date"])


In [30]:
df["date"].dt.day.value_counts()

date
3     2171
5     2167
1     2126
15    1897
13    1875
11    1875
2     1872
17    1835
6     1810
4     1807
16    1651
18    1624
12    1622
14    1614
26    1093
24    1088
9     1085
19    1076
21    1076
7     1075
30    1066
28    1065
10     933
20     930
25     917
29     913
8      906
31     903
23     902
27     898
Name: count, dtype: int64

In [24]:
df.reset_index(drop=True, inplace=True)

In [51]:
df["ordered_time"] = pd.to_datetime(df["ordered_time"], format="%H:%M:%S", errors="coerce").dt.time
df["ordered_time"].dtype

dtype('O')

In [57]:
df["picked_time"] = pd.to_datetime(df["picked_time"], errors="coerce").dt.time
df["picked_time"].isna().sum()

C:\Users\RR\AppData\Local\Temp\ipykernel_11040\1035933555.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["picked_time"] = pd.to_datetime(df["picked_time"], errors="coerce").dt.time


0

In [68]:
df["city_type"].value_counts()

city_type
Metropolitian    31355
Urban             9256
Semi-Urban         149
Name: count, dtype: int64

In [6]:
def time_of_day(col):

    return pd.cut(col, bins=[0,6,12,17,20,24], right=True, 
                 labels=["after_morning", "morning", "afternoon", "evening", "night"])

In [7]:
def calculate_distance(df):
    lat1 = df["rest_lat"]
    long1 = df["rest_long"]
    lat2 = df["delivery_lat"]
    long2 = df["delivery_long"]
    
    long1, lat1, long2, lat2 = map(np.radians, [long1, lat1, long2, lat2])
    
    dlong = long2 - long1
    dlat = lat2 - lat1
    
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlong / 2.0)**2
    
    c = 2 * np.arcsin(np.sqrt(a))
    
    distance = 6371 * c
    
    df["distance"] = distance

In [28]:
def calculate_pickup_time(df):

    picked_time = pd.to_datetime(df["picked_time"].astype(str))
    ordered_time = pd.to_datetime(df["ordered_time"].astype(str))

    return np.where(ordered_time < picked_time, 
             (picked_time - ordered_time).dt.seconds / 60, 
                np.NaN)

In [34]:
def clean_data(df):

    # drop useless columns:
    df.drop(columns=['Unnamed: 0', 'ID'], inplace=True)
    
    # rename the column names:
    df.rename(columns={"Delivery_person_ID": "id",
    "Delivery_person_Age":"age",
    "Delivery_person_Ratings":"ratings",
    "Restaurant_latitude": "rest_lat",
    "Restaurant_longitude": "rest_long",
    "Delivery_location_latitude": "delivery_lat",
    "Delivery_location_longitude": "delivery_long",
    "Order_Date": "date",
    "Time_Orderd": "ordered_time",
    "Time_Order_picked": "picked_time",
    "Weatherconditions": "weather",
    "Road_traffic_density": "traffic",
    "Vehicle_condition": "condition",
    "Type_of_order": "order_type",
    "Type_of_vehicle": "vehicle_type",
    "multiple_deliveries": "multi_deliveries",
    "Festival": "festival",
    "City": "city_type",
    "Time_taken(min)": "time"}, inplace=True)
    
    # fix some values
    df["weather"] = df["weather"].str.replace("conditions ", "").str.strip()
    df["time"] = df["time"].str.split(")").str.get(1).str.strip().astype(np.float32)
    df["id"] = df["id"].str.split("RES").str.get(0)
    df.rename(columns={"id": "city"}, inplace=True)

    # Strip categorical cols:
    cat_col = ["age", "ratings", "city", "weather", "traffic", "order_type", "vehicle_type", "multi_deliveries", "festival", "city_type"]

    for col in cat_col:
        df[col] = df[col].str.strip().str.lower()

    # remove NaN values:
    df.replace("NaN", np.nan, inplace=True)

    # Individually cleaning each columns:
    # age:
    df["age"] = df["age"].astype(np.float32)
    df.drop(index = df[df["age"] < 18].index, inplace=True)
   

    # ratings:
    df["ratings"] = df["ratings"].astype(np.float32)
    df.drop(index = df[df["ratings"] > 5].index, inplace=True)

    # reset_index:
    df.reset_index(drop=True, inplace=True)
    
    # convert date column to datetime:
    df["date"] = pd.to_datetime(df["date"])

    # Change ordered_time and picked_time to time object:
    df["ordered_time"] = pd.to_datetime(df["ordered_time"], format="%H:%M:%S", errors="coerce").dt.time
    df["picked_time"] = pd.to_datetime(df["picked_time"], errors="coerce").dt.time
    
    df["order_day"] = df["date"].dt.day
    df["order_month"] = df["date"].dt.month
    df["order_day_of_week"] = df["date"].dt.day_name().str.lower()
    df["is_weekend"] = df["date"].dt.day_name().isin(["Sunday", "Saturday"]).astype(int)
    df["order_time_hour"] = pd.to_datetime(df["ordered_time"], format="%H:%M:%S", errors="coerce").dt.hour
    df["order_time_of_day"] = time_of_day(df["order_time_hour"])
    df["pickup_time"] = calculate_pickup_time(df)

    # Distance and distance_type:
    calculate_distance(df)
    df["distance_type"] = pd.cut(df["distance"], [0, 5, 10, 15, 25],
                                right=False, labels=["short", "medium", "long", "very_long"])

    # Drop unnecessary columns:
    df.drop(columns = ["ordered_time", "picked_time"], inplace=True)

    print(df.info())

In [36]:
clean_data(df)

C:\Users\RR\AppData\Local\Temp\ipykernel_13300\4292215486.py:56: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["date"] = pd.to_datetime(df["date"])
C:\Users\RR\AppData\Local\Temp\ipykernel_13300\4292215486.py:60: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["picked_time"] = pd.to_datetime(df["picked_time"], errors="coerce").dt.time
C:\Users\RR\AppData\Local\Temp\ipykernel_13300\2975267094.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  picked_time = pd.to_datetime(df["picked_time"].astype(str))
C:\Users\RR\AppData\Local\Temp\ipykernel_13300\2975267094.py:4: UserWarning: Could not infer

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41872 entries, 0 to 41871
Data columns (total 26 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   city               41872 non-null  object        
 1   age                40153 non-null  float32       
 2   ratings            40109 non-null  float32       
 3   rest_lat           41872 non-null  float64       
 4   rest_long          41872 non-null  float64       
 5   delivery_lat       41872 non-null  float64       
 6   delivery_long      41872 non-null  float64       
 7   date               41872 non-null  datetime64[ns]
 8   weather            41872 non-null  object        
 9   traffic            41872 non-null  object        
 10  condition          41872 non-null  int64         
 11  order_type         41872 non-null  object        
 12  vehicle_type       41872 non-null  object        
 13  multi_deliveries   41872 non-null  object        
 14  festiv

In [32]:
outliers = df[df["pickup_time"] > 1000]

In [33]:
outliers.shape

(0, 28)

In [19]:
(pd.to_datetime(outliers["picked_time"].astype(str)) 
                          < pd.to_datetime(outliers["ordered_time"].astype(str))).shape

C:\Users\RR\AppData\Local\Temp\ipykernel_13300\2400226101.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  (pd.to_datetime(outliers["picked_time"].astype(str))
C:\Users\RR\AppData\Local\Temp\ipykernel_13300\2400226101.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  < pd.to_datetime(outliers["ordered_time"].astype(str))).shape


(79,)

In [27]:
calculate_pickup_time(df).shape

C:\Users\RR\AppData\Local\Temp\ipykernel_13300\3488331951.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  picked_time = pd.to_datetime(outliers["picked_time"].astype(str))
C:\Users\RR\AppData\Local\Temp\ipykernel_13300\3488331951.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  ordered_time = pd.to_datetime(outliers["ordered_time"].astype(str))


(79,)

In [17]:
pd.to_datetime(outliers["picked_time"].astype(str)) > pd.to_datetime(outliers["ordered_time"].astype(str))

C:\Users\RR\AppData\Local\Temp\ipykernel_13300\875258000.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(outliers["picked_time"].astype(str)) > pd.to_datetime(outliers["ordered_time"].astype(str))
C:\Users\RR\AppData\Local\Temp\ipykernel_13300\875258000.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(outliers["picked_time"].astype(str)) > pd.to_datetime(outliers["ordered_time"].astype(str))


32       False
1713     False
2454     False
2915     False
3308     False
3757     False
3799     False
4968     False
5391     False
5808     False
6747     False
6804     False
7925     False
8505     False
8866     False
9695     False
9715     False
10874    False
11270    False
11377    False
11992    False
13773    False
13955    False
14523    False
18342    False
18535    False
19840    False
20039    False
20082    False
20818    False
21440    False
21939    False
22128    False
22846    False
23238    False
23537    False
23586    False
24508    False
24791    False
24797    False
24840    False
24936    False
25107    False
25840    False
26338    False
27052    False
27173    False
27753    False
27997    False
28381    False
28567    False
29902    False
29999    False
30415    False
30895    False
31399    False
31879    False
32138    False
32353    False
33041    False
33371    False
33400    False
34872    False
35512    False
35612    False
36718    False
36842    F

In [7]:
# Extract date_time info from order_date:

df["order_day"] = df["date"].dt.day
df["order_month"] = df["date"].dt.month
df["order_day_of_week"] = df["date"].dt.day_name().str.lower()
df["is_weekend"] = df["date"].dt.day_name().isin(["Sunday", "Saturday"]).astype(int)
df["order_time_hour"] = pd.to_datetime(df["ordered_time"], format="%H:%M:%S", errors="coerce").dt.hour
df["order_time_of_day"] = time_of_day(df["order_time_hour"])
df["pickup_time"] = ((pd.to_datetime(df["picked_time"].astype(str)) 
                      - pd.to_datetime(df["ordered_time"].astype(str))).dt.seconds)/60

calculate_distance(df)

# Drop unnecessary columns:
df.drop(columns = ["ordered_time", "picked_time"])
df.head()

C:\Users\RR\AppData\Local\Temp\ipykernel_9776\1920066034.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["pickup_time"] = ((pd.to_datetime(df["picked_time"].astype(str))
C:\Users\RR\AppData\Local\Temp\ipykernel_9776\1920066034.py:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  - pd.to_datetime(df["ordered_time"].astype(str))).dt.seconds)/60


,city,age,ratings,rest_lat,rest_long,delivery_lat,delivery_long,date,ordered_time,picked_time,...,festival,city_type,time,order_day,order_month,order_day_of_week,is_weekend,order_time_hour,order_time_of_day,pickup_time
0,INDO,37.0,4.9,22.745049,75.892471,22.765049,75.912471,2022-03-19,11:33:33,11:45:29,...,No,Urban,24.0,19,3,saturday,1,11.0,morning,11.933333
1,BANG,34.0,4.5,12.913041,77.683237,13.043041,77.813237,2022-03-25,19:45:37,19:51:49,...,No,Metropolitian,33.0,25,3,friday,0,19.0,evening,6.200000
2,BANG,23.0,4.4,12.914264,77.678400,12.924264,77.688400,2022-03-19,08:32:58,08:48:47,...,No,Urban,26.0,19,3,saturday,1,8.0,morning,15.816667
3,COIMB,38.0,4.7,11.003669,76.976494,11.053669,77.026494,2022-04-05,18:03:58,18:12:52,...,No,Metropolitian,21.0,5,4,tuesday,0,18.0,evening,8.900000
4,CHEN,32.0,4.6,12.972793,80.249982,13.012793,80.289982,2022-03-26,13:34:16,13:45:36,...,No,Metropolitian,30.0,26,3,saturday,1,13.0,afternoon,11.333333


In [37]:
df.to_csv("../data/interim/India-Food-Delivery-Time-Prediction_cleaned.csv", index=False)

In [23]:
pd.read_csv("../data/interim/India-Food-Delivery-Time-Prediction_cleaned.csv").head(1)

,city,age,ratings,rest_lat,rest_long,delivery_lat,delivery_long,date,weather,traffic,...,time,order_day,order_month,order_day_of_week,is_weekend,order_time_hour,order_time_of_day,pickup_time,distance,distance_type
0,indo,37.0,4.9,22.745049,75.892471,22.765049,75.912471,2022-03-19,sunny,high,...,24.0,19,3,saturday,1,11.0,morning,11.933333,3.025149,short
